In [7]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

In [8]:
model = load_model("best_mask_model.h5") 

In [9]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

In [12]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# load model
model = load_model("best_mask_model.h5")

# face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

cap = cv2.VideoCapture(1)

if not cap.isOpened():
    print("❌ Camera not opened")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(gray, 1.1, 3)

    # 🔥 fallback if no face detected
    if len(faces) == 0:
        faces = [(100, 100, 300, 300)]

    for (x, y, w, h) in faces:

        # 🔥 padding for better crop
        pad = 20
        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(frame.shape[1], x + w + pad)
        y2 = min(frame.shape[0], y + h + pad)

        face = frame[y1:y2, x1:x2]

        # 🔥 BGR → RGB (MOST IMPORTANT FIX)
        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)

        # preprocess
        face = cv2.resize(face, (224, 224))
        face = face / 255.0
        face = np.reshape(face, (1, 224, 224, 3))

        # prediction
        pred = model.predict(face, verbose=0)
        confidence = pred[0][0]

        print("Confidence:", confidence)

        # 🔥 threshold adjust
        if confidence > 0.6:
            label = "No Mask"
            color = (0, 0, 255)
        else:
            label = "Mask"
            color = (0, 255, 0)

        # draw
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, label, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    cv2.imshow("Mask Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Confidence: 0.9979247
Confidence: 0.18195397
Confidence: 0.42983747
Confidence: 0.19134003
Confidence: 0.18997842
Confidence: 0.18997842
Confidence: 0.19036447
Confidence: 0.18565483
Confidence: 0.19033276
Confidence: 0.18063718
Confidence: 0.19680817
Confidence: 0.19689557
Confidence: 0.19069245
Confidence: 0.47091535
Confidence: 0.18036994
Confidence: 0.19675118
Confidence: 0.20182984
Confidence: 0.3276972
Confidence: 0.19100149
Confidence: 0.21861003
Confidence: 0.18474178
Confidence: 0.17932321
Confidence: 0.17321502
Confidence: 0.18648449
Confidence: 0.13088772
Confidence: 0.7268003
Confidence: 0.18436113
Confidence: 0.23509115
Confidence: 0.76037025
Confidence: 0.25886524
Confidence: 0.13106708
Confidence: 0.038599752
Confidence: 0.9948841
Confidence: 0.21638012
Confidence: 0.083899036
Confidence: 0.3027023
Confidence: 0.22319737
Confidence: 0.13402879
Confidence: 0.20273359
Confidence: 0.09795917
Confidence: 0.37219208
Confidence: 0.99857956
Confidence: 0.99963546
Confidence: 0.

In [11]:
faces = face_cascade.detectMultiScale(gray, 1.1, 3)
print("Faces detected:", len(faces))

if len(faces) == 0:
    print("⚠️ No face detected")
    faces = [(50, 50, 300, 300)]

for (x, y, w, h) in faces:
    face = frame[y:y+h, x:x+w]

    face = cv2.resize(face, (224,224))
    face = face / 255.0
    face = np.reshape(face, (1,224,224,3))

    pred = model.predict(face, verbose=0)
    print("Prediction:", pred[0][0])

    if pred[0][0] > 0.5:
        label = "No Mask"
        color = (0,0,255)
    else:
        label = "Mask"
        color = (0,255,0)

    cv2.rectangle(frame, (x,y), (x+w,y+h), color, 2)
    cv2.putText(frame, label, (x,y-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

Faces detected: 0
⚠️ No face detected
Prediction: 0.15958859
